# Holding speed through the break

Kaggle notebook for NFL Big Data Bowl 2027 (Combine to NFL Performance). The entry is this notebook plus a write-up, not a prediction CSV.

The notebook reads the nine official CSVs from `/kaggle/input/nfl-big-data-bowl-2027/` and does not download them. It draws Figures 1 and 2 at runtime. Do not export Figure 1 (one point per player) outside Kaggle.

Main lead, found after the dated amendments A1-A6 and still to be validated: rookie DBs who keep more speed through the backpedal break allow less EPA per credited target. Time to the break shows no link. The tight-end tests are weak. Definitions and amendments are in `src/bdb2027/preregistration.py`.

## Setup

Install this repository on the Kaggle notebook if `bdb2027` is not already importable. The next cell adds a local `src/` directory when the notebook is run from a checkout.


In [ ]:
import sys
from pathlib import Path

for candidate in (Path.cwd() / "src", Path.cwd(), Path("/kaggle/working/src")):
    if (candidate / "bdb2027").is_dir():
        sys.path.insert(0, str(candidate))
        break


In [ ]:
from pathlib import Path

from bdb2027.io import default_data_dir, missing_official_files
from bdb2027.pipeline import effects_summary, run_analysis
from bdb2027.preregistration import PREREGISTERED_TESTS, bonferroni_alpha

DATA_DIR = Path("/kaggle/input/nfl-big-data-bowl-2027")
if not DATA_DIR.is_dir():
    DATA_DIR = default_data_dir()

missing = missing_official_files(DATA_DIR)
if missing:
    raise FileNotFoundError(
        "Competition CSVs are not available at "
        f"{DATA_DIR}. Missing: {', '.join(missing)}. "
        "Accept the Kaggle rules and add the competition data. Do not commit the files."
    )
print("Data directory:", DATA_DIR)
print("Pre-registered tests:", len(PREREGISTERED_TESTS))
print("Bonferroni alpha:", round(bonferroni_alpha(), 4))


## Pre-registered family (after amendments)

Within-draft-class correlations with 95% Fisher intervals. The two DB break-time rows are within-class Spearman, with attempts censored at 5.5 s ranked slowest. The other rows are Pearson. The Bonferroni threshold is 0.05/6.

In [ ]:
result = run_analysis(DATA_DIR)
effects = result["effects"]
print(effects_summary(effects))

## Robustness of retention -> EPA allowed

Not pre-registered. Aggregates only. The write-up keeps the main row, the three year drops, the 10-target floor, and the man/zone split. This table also has the rank correlation, the bootstrap interval, and the range when any one player is removed.

In [ ]:
from bdb2027.io import load_games, load_players
from bdb2027.robustness import retention_robustness

players = load_players(DATA_DIR).drop_duplicates("nfl_id", keep="first")
games = load_games(DATA_DIR)
robustness = retention_robustness(result, players)
robustness.round(3)

## Probabilities and scenarios

This is a within-class bootstrap approximation to a posterior with a flat prior. Bands are 90%. The scenarios compare DBs at the within-class 75th and 25th percentiles of retention. "Season" uses an assumed 60 targets for a full-time starter, which is not a count from these files. The observed credited volume is draftee-on-draftee only. Expect these effect sizes to shrink on new data.

In [ ]:
from bdb2027.scenarios import scenario_report

scenarios = scenario_report(result, players, games)
scenarios.round(3)

## Alignment check (cushion)

Are DBs with no break by 5.5 s lined up deeper before the snap? `cushion` is the published pre-snap distance to the receiver. The within-class gap is censored minus detected, with a within-class permutation p.

In [ ]:
from bdb2027.alignment import cushion_check, load_db_cushion

db_players = result["db_players"]
db_ids = set(db_players.loc[db_players["db_break_time_s"].notna(), "nfl_id"])
cushion_rows = load_db_cushion(DATA_DIR, db_ids, games)
cushion = cushion_check(cushion_rows, db_players, players, result["assignments"])
cushion.round(3)

## Figure 1

Within-class deviations of speed retention against EPA allowed per credited target. The line is the pooled within-class slope.

In [ ]:
import matplotlib.pyplot as plt

from bdb2027.figures import figure_forest, figure_retention_scatter

figure_retention_scatter(result["db_players"])
plt.show()

## Figure 2

Intervals for the six family tests (Fisher) and the retention robustness checks. For the leave-one-player-out row, the bar is the range of r.

In [ ]:
figure_forest(effects, robustness)
plt.show()

## Optional external benchmark

Leave this cell unchanged unless the rating file is publicly available to every participant at no cost, or it meets the host Reasonableness Standard. Madden ratings were not verified to meet that test. Do not scrape ratings and do not commit them.

Set `ENABLE_EXTERNAL_BENCHMARK` to `True` and point `RATING_CSV` at a local file with `nfl_id` and any of: `speed`, `acceleration`, `agility`, `change_of_direction`, `route_running`, `release`, `man_coverage`, `zone_coverage`.


In [ ]:
ENABLE_EXTERNAL_BENCHMARK = False
RATING_CSV = None  # path outside the repo and outside the competition CSVs

external = run_analysis(
    DATA_DIR,
    external_benchmark_path=None if RATING_CSV is None else Path(RATING_CSV),
    external_benchmark_enabled=ENABLE_EXTERNAL_BENCHMARK,
)["external"]
print(external["status"])
print(external["reason"])
if external["status"] == "computed":
    display(external["comparisons"])


## Sample sizes

Replace the planning notes in the write-up with these counts. Do not treat a planning guess of about 18 tight ends or about 62 defensive backs as a measured sample.


In [ ]:
print("Games loaded:", result["n_games"])
print("TE players in the merge:", len(result["te_players"]))
print("DB players in the merge:", len(result["db_players"]))
print(effects.loc[:, ["test_id", "n_players", "n_classes", "class_counts", "small_n"]])


## Exploratory cornerbacks

Not pre-registered, and not part of the family of six. The DB tests above still pool CB, SS, FS, DB, and S. This section keeps `nfl_position == CB` only.

Man versus zone uses the corner's `coverage_assignment`. `team_coverage_man_zone` is the defensive shell for every defender on the play, so it is reported only as an agreement rate. A threat is a completion on the credited target. Kinematic features stop at `pass_forward`. The model is penalized logistic regression with a player intercept, plus wing span, broad jump, and vertical.

The coverage figure has one point per CB. Do not export it outside Kaggle, and do not paste the player table into the write-up without checking the figure budget. Lower `N_BOOTSTRAP` for a faster rerun. The default 200 matches the CLI.


In [ ]:
from bdb2027.figures import figure_cb_coverage, figure_threat_effects
from bdb2027.pipeline import run_cb_exploratory

N_BOOTSTRAP = 200
cb = run_cb_exploratory(
    DATA_DIR,
    assignments=result["assignments"],
    players=players,
    n_bootstrap=N_BOOTSTRAP,
)
print(cb["coverage_summary"].T)
print(cb["coverage_audit"].to_string(index=False))
print(cb["threat_metrics"].T)
effects = cb["threat_effects"]
if effects.empty:
    print("Threat model was not fit. See threat_metrics for the sample size.")
else:
    columns = ["term", "coef", "ci_low", "ci_high", "odds_ratio", "mean_risk_difference"]
    print(effects.loc[:, columns].to_string(index=False))
print(cb["threat_calibration"].to_string(index=False))


## Exploratory figures

Blue is man, orange is zone. The bar is the plugged-in 95% posterior interval for shrunk EPA allowed. The forest is the threat-model log-odds per 1 SD. The intercept is omitted. Blank intervals mean fewer than 20 bootstrap fits succeeded.

The cell above prints every coefficient and the calibration bins. The write-up reports completion-chance changes only for slopes whose intervals exclude zero, and it leaves the bin-by-bin calibration here.


In [ ]:
shown = cb["coverage"].loc[cb["coverage"]["meets_floor"].astype(bool)]
print(f"CB coverage rows meeting the 5-target floor: {len(shown)}")
figure_cb_coverage(cb["coverage"])
plt.show()
figure_threat_effects(cb["threat_effects"])
plt.show()
